# 01 — Kiva Calibration

Maps the public Kiva snapshot to `MicroLoanPricing-v0` simulator priors.

**Outputs:** `configs/calibration_kiva.yaml` and a one-page `calibration_report.md`.

See `docs/calibration.md` for the full protocol.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('../data/raw/kiva_loans.csv')
OUT  = Path('../configs/calibration_kiva.yaml')

assert DATA.exists(), f'Place the Kiva snapshot at {DATA}'
df = pd.read_csv(DATA)
print(df.shape)
df.head()

## 1. Filter

In [ ]:
STATUS = {'paid', 'defaulted', 'refunded'}
COUNTRIES = {'KE','IN','BD','PH','GT','MX'}

f = df[df['status'].isin(STATUS) & df['country_code'].isin(COUNTRIES)]
f = f[(f['loan_amount'] >= 25) & (f['loan_amount'] <= 2000)]
f = f.dropna(subset=['sector'])
len(f)

## 2. Fit reservation rate (Beta)

In [ ]:
from scipy import stats

# Use funding_success as a noisy proxy for acceptance at the offered yield.
# In a real fit you would invert the Kiva auction mechanics here.
if 'funded_amount' in f.columns and 'loan_amount' in f.columns:
    funded_frac = (f['funded_amount'] / f['loan_amount']).clip(0, 1)
    alpha, beta_param, *_ = stats.beta.fit(funded_frac.dropna(), floc=0, fscale=1)
    print(f'Beta(α={alpha:.3f}, β={beta_param:.3f})')
else:
    alpha, beta_param = 2.0, 8.0  # fallback

## 3. Default base rate, per sector

In [ ]:
f['defaulted'] = (f['status'] == 'defaulted').astype(int)
by_sector = f.groupby('sector')['defaulted'].mean().sort_values()
by_sector

## 4. Write the override YAML

In [ ]:
import yaml
out = {
    'provenance': {
        'source': 'kiva-snapshot',
        'snapshot_date': str(pd.Timestamp.now().date()),
    },
    'cohort': {
        'reservation_rate_alpha': float(alpha),
        'reservation_rate_beta':  float(beta_param),
        'default_prob_base': float(by_sector.median()),
    },
    'sector_default_base': by_sector.to_dict(),
}
OUT.write_text(yaml.dump(out, sort_keys=False))
print(f'Wrote {OUT}')